<a href="https://colab.research.google.com/github/jhughes7386/cosc-650-applied-llm-systems/blob/week-8-reading-notes-clean/week-08/week8_capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 (starter): Integrated LLM System (capstone)

A small skeleton that integrates three techniques you already built: retrieval (Week 5), a tool with routing (Week 4), and a grounded prompt (Week 3). Orchestration runs locally with no key; synthesis and the LLM-as-judge are guarded. **Reuse your own components**; this is a starting frame, not the finished system. Cells marked **TODO (you)** are yours.

## Part 1: Integrate


### 1.1 RAG
The RAG component uses Amazon's 2024 and 2025 annual reports.

Document Chunking: Each page's text is split into fixed-size chunks of 320 characters with a 40-character overlap between consecutive chunks. The overlap helps preserve context between chunks. Each chunk retains its report name and page number so retrieved information can be traced back to its source.

The chunks are embedded using sentence-transformers/all-MiniLM-L6-v2 and stored in a FAISS index for similarity-based retrieval.

In [1]:
# load Amazon financial reports from drive
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Datasets")

pdf_paths = [
    DATA_DIR / "Amazon-2024-Annual-Report.pdf",
    DATA_DIR / "Amazon-2025-Annual-Report.pdf",
]

Mounted at /content/drive


In [2]:
# install modules silently
%%capture
%pip install -q faiss-cpu pypdf

In [3]:
import os, re, ast, operator, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
from pypdf import PdfReader
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Technique 1: RAG retrieval
# Adapted from Week 5: document loading, chunking, embeddings, FAISS.

def fetch_doc(pdf_path):
    """Extract text from a PDF, preserving page boundaries."""
    pages = []

    reader = PdfReader(str(pdf_path))

    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = re.sub(r'\s+', ' ', text).strip()

        if text:
            pages.append({
                "text": text,
                "page": page_num
            })

    return pages


# Load the two Amazon annual reports.
DOCS = {
    path.name: fetch_doc(path)
    for path in pdf_paths
}

for name, pages in DOCS.items():
    print(
        f"{name:35s} -> "
        f"{len(pages)} pages with text, "
        f"{sum(len(p['text']) for p in pages):,} characters"
    )


def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip()
    out, i = [], 0

    while i < len(text):
        out.append(text[i:i + size])
        i += size - overlap

    return out

# Chunking configuration
chunks = []

for name, pages in DOCS.items():
    for page in pages:
        page_chunks = chunk_fixed(page["text"], 320, 40)

        for chunk in page_chunks:
            chunks.append({
                "report": name,
                "page": page["page"],
                "text": chunk
            })

print(f"\nTotal chunks: {len(chunks)}")


def build(chunks):
    # Embed only the text portion of each chunk.
    embeddings = embedder.encode(
        [chunk["text"] for chunk in chunks],
        normalize_embeddings=True
    ).astype("float32")

    idx = faiss.IndexFlatIP(embeddings.shape[1])
    idx.add(embeddings)

    return idx


index = build(chunks)


def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode(
        [q],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = idx.search(qe, min(k, len(chunks)))

    return [
        {
            **chunks[i],
            "score": float(score)
        }
        for score, i in zip(scores[0], indices[0])
        if i >= 0
    ]


print("Amazon RAG index ready.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Amazon-2024-Annual-Report.pdf       -> 90 pages with text, 320,235 characters
Amazon-2025-Annual-Report.pdf       -> 91 pages with text, 324,109 characters

Total chunks: 2393
Amazon RAG index ready.


### 1.2 Dynamic few-shot

Dynamic few-shot uses the existing embedding model to select the two most relevant question-answer examples for each new question. These examples are added to the prompt to guide Gemini Flash's response, while RAG retrieves supporting evidence from Amazon's annual reports.

In [4]:
# Technique 2: Dynamic few-shot adaptation
# Example question-answer pairs to guide responses
FEW_SHOT_EXAMPLES = [
    {
        "question": "What was Amazon's total revenue in 2025?",
        "answer": "Amazon's total revenue increased from $638 billion in 2024 to $717 billion in 2025."
    },
    {
        "question": "How much revenue did AWS generate in 2024?",
        "answer": "AWS revenue increased from $91 billion in 2023 to $108 billion in 2024."
    },
    {
        "question": "What was Amazon's operating income in 2024?",
        "answer": "Amazon's operating income increased from $36.9 billion in 2023 to $68.6 billion in 2024."
    },
    {
        "question": "What was Amazon's operating income in 2025?",
        "answer": "Amazon's operating income increased from $69 billion in 2024 to $80 billion in 2025."
    },
    {
        "question": "How did North America revenue change in 2025?",
        "answer": "North America revenue increased from $387 billion in 2024 to $426 billion in 2025, a 10% year-over-year increase."
    },
    {
        "question": "How did International revenue change in 2025?",
        "answer": "International revenue increased from $143 billion in 2024 to $162 billion in 2025, a 13% year-over-year increase."
    },
    {
        "question": "What happened to Amazon's free cash flow in 2025?",
        "answer": "Amazon's free cash flow decreased from $38 billion in 2024 to $11 billion in 2025. The report attributes the decrease primarily to increased property and equipment purchases."
    },
    {
        "question": "What was AWS revenue in 2025?",
        "answer": "AWS revenue increased from $108 billion in 2024 to $129 billion in 2025, a 20% year-over-year increase."
    }
]

# Embed the example questions for similarity-based selection
few_shot_embeddings = embedder.encode(
    [example["question"] for example in FEW_SHOT_EXAMPLES],
    normalize_embeddings=True
).astype("float32")

few_shot_index = faiss.IndexFlatIP(few_shot_embeddings.shape[1])
few_shot_index.add(few_shot_embeddings)

def retrieve_examples(question, k=2):
    query_embedding = embedder.encode(
        [question],
        normalize_embeddings=True
    ).astype("float32")

    _, indices = few_shot_index.search(
        query_embedding,
        min(k, len(FEW_SHOT_EXAMPLES))
    )

    return [FEW_SHOT_EXAMPLES[i] for i in indices[0] if i >= 0]

print("Components ready: RAG retrieval, dynamic few-shot example selection")

Components ready: RAG retrieval, dynamic few-shot example selection


### 1.3 Tool Use

The tool is a calculator for basic operations, such as calculating year-over-year revenue changes. This allows the system to perform calculations rather than relying on the LLM to do the math.

In [5]:
# Technique 3: Tool use
import ast
import operator

_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}

def _ev(n):
    if isinstance(n, ast.Constant) and type(n.value) in (int, float):
        return n.value
    if isinstance(n, ast.BinOp) and type(n.op) in _OPS:
        return _OPS[type(n.op)](_ev(n.left), _ev(n.right))
    raise ValueError("Arithmetic only")

def calculator(expr):
    return _ev(ast.parse(expr, mode="eval").body)

print("Tool ready: calculator")
print(calculator("717 - 638"))

Tool ready: calculator
79


## 1.4 Combine into one System

In [18]:
def gemini_chat(messages, model="gemini-3.5-flash-lite"):
    if not os.environ.get("GEMINI_API_KEY"):
        return None

    from openai import OpenAI

    client = OpenAI(
        api_key=os.environ["GEMINI_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    return response.choices[0].message.content
def assistant(q, tool_expr=None):
    ctx = retrieve(index, chunks, q, k=3)
    examples = retrieve_examples(q, k=2)
    tool = calculator(tool_expr) if tool_expr else None

    context_text = "\n- ".join(
        f"[{r['report']}, page {r['page']}] {r['text']}"
        for r in ctx
    )

    examples_text = "\n".join(
        f"Question: {e['question']}\nAnswer: {e['answer']}"
        for e in examples
    )

    prompt = (
        "Answer the question using the retrieved report evidence "
        "and calculator result, if provided. Use the examples as "
        "guidance, not as factual evidence. Do not invent facts. "
        "Cite report names and page numbers.\n"
        f"Context:\n- {context_text}\n"
        f"Examples:\n{examples_text}\n"
        + (f"Tool result: {tool}\n" if tool is not None else "")
        + f"Question: {q}\nAnswer:"
    )

    out = gemini_chat([{"role": "user", "content": prompt}])

    return {
        "context": ctx,
        "examples": examples,
        "tool": tool,
        "answer": out if out is not None else
            "[API-BLOCKED] Grounded prompt built; set GEMINI_API_KEY to synthesize."
    }


print(assistant("What was Amazon's AWS revenue in 2025?"))
print(assistant(
    "How much did Amazon's total revenue increase from 2024 to 2025?",
    tool_expr="717 - 638"
))


{'context': [{'report': 'Amazon-2024-Annual-Report.pdf', 'page': 2, 'text': 'Dear Shareholders: 2024 was a strong year for Amazon. Our total revenue grew 11% year-over-year (“Y oY”) from $575B to $638B. By segment, North America revenue increased 10% Y oY from $353B to $387B, International revenue grew 9% Y oY from $131B to $143B, and A WS revenue increased 19% Y oY , from $91B to $108B. For pe', 'score': 0.7878559231758118}, {'report': 'Amazon-2025-Annual-Report.pdf', 'page': 8, 'text': 'Parting thoughts. Amazon had another strong year in 2025. Amazon’s revenue in 2025 grew 12% year-over-year (“Y oY”) from $638 billion to $717 billion. By segment, North America revenue increased 10% Y oY from $387 billion to $426 billion, International revenue grew 13% Y oY from $143 billion to $162 billion, and A WS r', 'score': 0.7717273235321045}, {'report': 'Amazon-2025-Annual-Report.pdf', 'page': 8, 'text': '$143 billion to $162 billion, and A WS revenue increased 20% Y oY , from $108 billion to 

## Part 2: Evaluate (model-free metrics + guarded judge)
Routing accuracy and answer-grounding are measurable without the model. The LLM-as-judge pass is guarded.

### 2.1: Report Eval Scores

In [19]:
# Question, expected route, expected terms, calculator expression, expected result
evalset = [
    ("Total revenue in 2025?", "rag", ["717"], None, None),
    ("AWS revenue in 2024?", "rag", ["108"], None, None),
    ("Operating income in 2024?", "rag", ["68.6", "68,593"], None, None),
    ("Operating income in 2025?", "rag", ["80", "79,975"], None, None),
    ("AWS revenue in 2025?", "rag", ["129", "128,725"], None, None),
    ("How did North America revenue change in 2025?", "rag", ["426", "387"], None, None),
    ("What happened to free cash flow in 2025?", "rag", ["38", "11"], None, None),
    ("What caused the decrease in free cash flow?", "rag", ["property", "equipment"], None, None),
    ("Revenue increase from 2024 to 2025?", "tool", ["638", "717"], "717 - 638", 79),
    ("AWS percentage revenue increase?", "tool", ["108", "129"],
     "(129 - 108) / 108 * 100", 19.4444444444)
]

def route_question(q):
    q = q.lower()
    return "tool" if any(
        word in q for word in ["calculate", "percentage", "difference", "how many"]
    ) else "rag"

route_ok = 0
retrieval_ok = 0
tool_ok = 0
tool_count = 0

for q, route, terms, expr, expected in evalset:
    route_ok += route_question(q) == route

    results = retrieve(index, chunks, q, k=5)
    context = " ".join(r["text"] for r in results).lower()
    retrieval_ok += any(term.lower() in context for term in terms)

    if expr is not None:
        tool_count += 1
        tool_ok += abs(calculator(expr) - expected) < 0.01

print(f"Routing accuracy: {route_ok / len(evalset):.0%}")
print(f"Retrieval evidence match rate: {retrieval_ok / len(evalset):.0%}")
print(f"Calculator accuracy: {tool_ok / tool_count:.0%}")

Routing accuracy: 90%
Retrieval evidence match rate: 80%
Calculator accuracy: 100%


### 2.2: LLM Judge

Create function to judge the LLM

In [20]:
def judge_answer(question, answer, context, tool_result=None):
    prompt = f"""
Score this answer from 1 to 5 on:
- correctness: Is the answer factually correct?
- grounding: Is it supported by the provided evidence?
- citations: Are report citations appropriate?

Question: {question}
Evidence: {context}
Calculator result: {tool_result}
Answer: {answer}

Return only JSON:
{{"correctness": 1, "grounding": 1, "citations": 1}}
"""
    response = gemini_chat([{"role": "user", "content": prompt}])

    if not response:
        print("Judge returned an empty response.")
        return None

    try:
        # Handle JSON wrapped in Markdown code fences
        cleaned = response.strip()
        if cleaned.startswith("```"):
            cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", cleaned)

        scores = json.loads(cleaned)
        dimensions = ["correctness", "grounding", "citations"]

        if all(
            isinstance(scores.get(d), (int, float))
            and not isinstance(scores.get(d), bool)
            and 1 <= scores[d] <= 5
            for d in dimensions
        ):
            return scores

        print("Judge response is missing valid scores:", response)

    except (ValueError, TypeError):
        print("Judge response was not valid JSON:", response)

    return None


Setup the LLM Judge
- run 1 at a time up to 10 to preserve quota
- each case cell uses the API calls

In [21]:
import json
import os
from google.colab import userdata

try:
    api_key = userdata.get("GEMINI_API_KEY")
    if api_key:
        os.environ["GEMINI_API_KEY"] = api_key
except Exception:
    pass

# Start a fresh evaluation.
judge_results = []

with open("judge_results.json", "w") as f:
    json.dump(judge_results, f, indent=2)

def run_judge_case(i):
    if not os.environ.get("GEMINI_API_KEY"):
        print("Missing GEMINI_API_KEY. Stopping.")
        return

    q, route, terms, expr, expected = evalset[i]

    print(f"Running case {i + 1}/10: {q}")

    try:
        result = assistant(q, tool_expr=expr)
        answer = result["answer"]

        if not answer or answer.startswith("[API-BLOCKED]"):
            print("No answer returned. Case not saved.")
            return

        context = "\n".join(
            f"[{r['report']}, page {r['page']}] {r['text']}"
            for r in result["context"]
        )

        scores = judge_answer(
            question=q,
            answer=answer,
            context=context,
            tool_result=result["tool"]
        )

        if scores is None:
            print("No valid judge scores. Case not saved.")
            return

        record = {
            "question": q,
            "expected_route": route,
            "expected_terms": terms,
            "expected_result": expected,
            "answer": answer,
            "evidence": context,
            "tool_result": result["tool"],
            "scores": scores
        }

        judge_results.append(record)

        # Save immediately after each successful case.
        with open("judge_results.json", "w") as f:
            json.dump(judge_results, f, indent=2)

        print("Saved successfully.")
        print("Cases saved:", len(judge_results))
        print("Scores:", scores)

    except Exception as e:
        # Preserve all earlier successful results.
        with open("judge_results.json", "w") as f:
            json.dump(judge_results, f, indent=2)

        if "429" in str(e) or "quota" in str(e).lower():
            print("QUOTA ERROR. Stop running cases until the quota resets.")
        else:
            print("Case failed:", str(e))

        print("Previously saved cases:", len(judge_results))


In [22]:
# Case 1
run_judge_case(0)

Running case 1/10: Total revenue in 2025?
Saved successfully.
Cases saved: 1
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}


In [23]:
# Case 2
run_judge_case(1)

Running case 2/10: AWS revenue in 2024?
Saved successfully.
Cases saved: 2
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}


In [24]:
# Case 3
run_judge_case(2)

Running case 3/10: Operating income in 2024?
Saved successfully.
Cases saved: 3
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}


In [25]:
# Case 4
run_judge_case(3)

Running case 4/10: Operating income in 2025?
Saved successfully.
Cases saved: 4
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}


In [26]:
# Case 5
run_judge_case(4)

Running case 5/10: AWS revenue in 2025?
Saved successfully.
Cases saved: 5
Scores: {'correctness': 2, 'grounding': 3, 'citations': 3}


In [27]:
# Case 6
run_judge_case(5)

Running case 6/10: How did North America revenue change in 2025?
Saved successfully.
Cases saved: 6
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}


In [28]:
# Case 7
run_judge_case(6)

Running case 7/10: What happened to free cash flow in 2025?
Saved successfully.
Cases saved: 7
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}


In [29]:
# Case 8
run_judge_case(7)

Running case 8/10: What caused the decrease in free cash flow?
Saved successfully.
Cases saved: 8
Scores: {'correctness': 2, 'grounding': 3, 'citations': 3}


In [30]:
# Case 9
run_judge_case(8)

Running case 9/10: Revenue increase from 2024 to 2025?
Saved successfully.
Cases saved: 9
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}


In [31]:
# Case 10
run_judge_case(9)

Running case 10/10: AWS percentage revenue increase?
Saved successfully.
Cases saved: 10
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}


In [32]:
# This cell downloads saved results, but evaluates and prints only the last 10.
from google.colab import files

with open("judge_results.json", "r") as f:
    saved_results = json.load(f)

last_10 = saved_results[-10:]

print(f"Total saved cases: {len(saved_results)}")
print(f"Cases evaluated in this summary: {len(last_10)}")

if last_10:
    for dimension in ["correctness", "grounding", "citations"]:
        scores = [
            r["scores"][dimension]
            for r in last_10
            if dimension in r.get("scores", {})
        ]

        if scores:
            print(
                f"{dimension.title()} score: "
                f"{sum(scores) / len(scores):.2f}/5 "
                f"(n={len(scores)})"
            )

    print("\nLast 10 Runs:")
    for i, result in enumerate(
        last_10, start=len(saved_results) - len(last_10) + 1
    ):
        print(f"\n--- Result {i} ---")
        print("Question:", result.get("question", "Not saved"))
        print("Model:", result.get("model", "Not recorded"))
        print("Answer:", result.get("answer", "Not saved"))
        print("Scores:", result.get("scores", "Not saved"))

    # Download the complete JSON, including results outside the last 10.
    files.download("judge_results.json")
else:
    print("No successful cases were saved.")


Total saved cases: 10
Cases evaluated in this summary: 10
Correctness score: 4.40/5 (n=10)
Grounding score: 4.60/5 (n=10)
Citations score: 4.60/5 (n=10)

Last 10 Runs:

--- Result 1 ---
Question: Total revenue in 2025?
Model: Not recorded
Answer: Based on the provided context, there is no mention of Amazon's total revenue for 2025.
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}

--- Result 2 ---
Question: AWS revenue in 2024?
Model: Not recorded
Answer: AWS revenue increased from $91 billion in 2023 to $108 billion in 2024 (Amazon-2024-Annual-Report.pdf, page 2).
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}

--- Result 3 ---
Question: Operating income in 2024?
Model: Not recorded
Answer: Amazon's consolidated operating income in 2024 was $68,593 million (or $68.6 billion) ([Amazon-2025-Annual-Report.pdf, page 38]).
Scores: {'correctness': 5, 'grounding': 5, 'citations': 5}

--- Result 4 ---
Question: Operating income in 2025?
Model: Not recorded
Answer: Amazon

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Part 2: Evaluation

I evaluated the assistant using routing, retrieval, calculator, and LLM-as-judge metrics.

### Component Metrics

| Metric | Result |
|---|---:|
| Routing accuracy | 90% |
| Retrieval evidence match rate | 80% |
| Calculator accuracy | 100% |

The calculator achieved 100% accuracy on the tested calculations. Routing accuracy was 90%, while retrieval evidence matched the expected evidence in 80% of the test cases. This shows there is still room to improve routing and retrieval.

### LLM-as-Judge Evaluation

I evaluated 10 held-out questions using Gemini. The judge scored each answer from 1 to 5 across three dimensions.

| Dimension | Average Score |
|---|---:|
| Correctness | 4.40/5 |
| Grounding | 4.60/5 |
| Citation quality | 4.60/5 |

The assistant performed well overall, especially when answering questions where clear evidence was available in the retrieved annual reports. However, some questions exposed limitations. For example, the assistant did not provide the correct 2025 AWS revenue and did not fully explain the decrease in free cash flow. These results show that strong grounding and citations do not always guarantee a complete or correct answer.

The evaluation also showed that the assistant sometimes reported that information was unavailable when the retrieved context did not contain the needed numbers. While this is preferable to inventing answers, improving retrieval and how the assistant uses the retrieved evidence would help it answer more questions accurately.

## Part 3 and 4: failure, submit, curate
**TODO (you):** show one failure of the integrated system, trace it to a component, and mitigate. Write a README documenting the architecture and technique choices, then curate the repo as your portfolio piece. Rubric: integration by reuse (30), evaluation incl judge (25), failure traced (15), README (20), portfolio curation (10).

### Part 3: Failure Cases

#### Part 3.1: Direct vs Indirect Questions


In [33]:
questions = [
    "What drove Amazon's increased investment in property and equipment?",
    "What were Amazon's capital expenditures in 2025, and what were they used for?"
]

for q in questions:
    results = retrieve(index, chunks, q, k=5)

    print("\n" + "=" * 80)
    print("Question:", q)

    for i, r in enumerate(results, 1):
        print(f"\n{i}. [{r['report']}, page {r['page']}]")
        print(r["text"])



Question: What drove Amazon's increased investment in property and equipment?

1. [Amazon-2024-Annual-Report.pdf, page 2]
Dear Shareholders: 2024 was a strong year for Amazon. Our total revenue grew 11% year-over-year (“Y oY”) from $575B to $638B. By segment, North America revenue increased 10% Y oY from $353B to $387B, International revenue grew 9% Y oY from $131B to $143B, and A WS revenue increased 19% Y oY , from $91B to $108B. For pe

2. [Amazon-2024-Annual-Report.pdf, page 80]
ority allocated to the AWS segment. Total net additions to property and equipment include the effect of non-cash activity such as property and equipment acquired but not yet paid for. Total net additions to property and equipment are as follows (in millions): Year Ended December 31, 2022 2023 2024 North America (1) $ 2

3. [Amazon-2025-Annual-Report.pdf, page 11]
Infrastructure During 1997, we worked hard to expand our business infrastructure to support these greatly increased traffic, sales, and service l

##### Mitigation: Direct vs. Indirect Questions

The RAG system struggles to retrieve relevant chunks for the question, "What drove Amazon's increased investment in property and equipment?" The top five results contain revenue figures and historical information instead of explanations incldued in the documents.

I tested a more direct question about Amazon's 2025 capital expenditures. This retrieved a relevant passage from page 34 of the 2025 annual report, explaining that the investments primarily supported AWS technology infrastructure and additional fulfillment network capacity.

This suggests that retrieval works best with direct questions instead of indirectly phrased questions. The limitation is about retrieval relevance rather than missing information.

**Possible mitigation:** There are a few options to test, including reranking retrieved passages, use larger chunks to preserve context, or rewrite indirect questions into more specific queries about capital expenditures, AWS infrastructure, and fulfillment capacity.

#### Part 3.2: Calculator Selection



In [34]:
q = "What was the revenue increase from 2024 to 2025?"

# 1. Test the routing heuristic
expected_route = "tool"
predicted_route = route_question(q)

print("Question:", q)
print("\n1. Routing")
print("Expected:", expected_route)
print("Predicted:", predicted_route)
print("Routing failure:", predicted_route != expected_route)

# 2. Check whether RAG retrieves the needed evidence
results = retrieve(index, chunks, q, k=3)
evidence_found = any(
    "638" in r["text"] and "717" in r["text"]
    for r in results
)

print("\n2. Retrieval")
for i, r in enumerate(results, 1):
    print(f"{i}. [{r['report']}, page {r['page']}] {r['text'][:200]}")
print("Both revenue figures retrieved:", evidence_found)

# 3. Compare calculator use with and without an expression
without_tool = assistant(q)
with_tool = assistant(q, tool_expr="717 - 638")

print("\n3. Calculator")
print("Without explicit expression:", without_tool["tool"])
print("With explicit expression:", with_tool["tool"])


Question: What was the revenue increase from 2024 to 2025?

1. Routing
Expected: tool
Predicted: rag
Routing failure: True

2. Retrieval
1. [Amazon-2025-Annual-Report.pdf, page 8] Parting thoughts. Amazon had another strong year in 2025. Amazon’s revenue in 2025 grew 12% year-over-year (“Y oY”) from $638 billion to $717 billion. By segment, North America revenue increased 10% Y
2. [Amazon-2025-Annual-Report.pdf, page 38] General and Administrative General and administrative costs in 2025 did not significantly change compared to the prior year. Other Operating Expense (Income), Net Other operating expense (income), net
3. [Amazon-2025-Annual-Report.pdf, page 62] earned revenue as of December 31, 2024 was $24.6 billion, of which $17.4 billion was recognized as revenue during the year ended December 31, 2025 and our total unearned revenue as of December 31, 202
Both revenue figures retrieved: True

3. Calculator
Without explicit expression: None
With explicit expression: 79


##### Mitigation: Calculator Selection

The question "What was the revenue increase from 2024 to 2025?" was expected to use the calculator, but the routing heuristic chose RAG instead. The heuristic checks for keywords such as "calculate," "percentage," "difference," and "how many," but does not recognize "increase."

The retrieval results show that RAG found both revenue figures (638 billion in 2024 and 717 billion in 2025). The calculator returned `None` when no expression was supplied, but returned `79` when given `717 - 638`. The limitation is that the integrated assistant requires the calculator expression to be passed in manually instead of automatically selecting the tool.

**Possible mitigations:**

1. **Expand the keyword rules:** Add key words such as "increase," "decrease," "growth," and "change" to the routing heuristic. This is the simplest fix, but it may also use the calculator when they only ask for a qualitative explanation. It also requires maintaining a list of keywords.

2. **Use the LLM to select the tool:** Update the system instructions to tell the model when to use the calculator, and have it return a structured tool choice and expression. The code would then validate the expression, execute the calculator, and provide the result to the model for its final answer. This is more flexible than keyword matching, but requires additional implementation, model calls, and expression validation.

For this project, expanding the keywords is the simpler mitigation, but LLM-based tool selection would be better if the system needed to handle a variety of questions. The current test demonstrates the LLM's routing limitation, but it does not produce an incorrect final answer.